In [1]:
%load_ext autoreload
%autoreload 2

# Part 2: Content-Base Filtering with Ridge Regression
In this notebook, the mission is implementing a Content-Base Filtering Recommender System by treating each user as a separate machine learning problem to learn their preferences for different genres

**Methods to learn:**
* **Items Profile: ($X$):** Binary genres vectors
* **Users Profile: ($W$):** Learned weights represents genre preferences
* **Target ($Y$):** The ratings given by users
* **Regularization:** Using **Ridge Model** to prevent overfitting especially users who gave few ratings

## Import libraries

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge 
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

import sys
sys.path.append('../api')
from src.recommender import ContentBasedFiltering

## 1. Import files

### 1.1.Train-Test dataset

In [3]:
ratings_train=pd.read_csv('../data/processed/ratings_a_train.csv').values
ratings_test=pd.read_csv('../data/processed/ratings_a_test.csv').values

In [4]:
ratings_train

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       ...,
       [      942,      1187,         3, 888640250],
       [      942,      1227,         3, 888640275],
       [      942,      1329,         3, 888692465]], shape=(90570, 4))

In [5]:
print(f'Number of traing rates: {ratings_train.shape[0]}')
print(f'Number of test rates: {ratings_test.shape[0]}')

Number of traing rates: 90570
Number of test rates: 9430


### 1.2.Items Profile

In [6]:
X_items_profile=np.load('../data/processed/items_profile.npy')

In [7]:
X_items_profile

array([[0, 0, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1682, 18))

## 2. Build Content-Based Recommender System

### 2.1.Build features vector

In [8]:
transfomer=TfidfTransformer(smooth_idf=True,norm='l2')
tfidf=transfomer.fit_transform(X_items_profile.tolist()).toarray()

In [9]:
tfidf

array([[0.        , 0.        , 0.74066017, ..., 0.        , 0.        ,
        0.        ],
       [0.53676706, 0.65097024, 0.        , ..., 0.53676706, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(1682, 18))

### 2.2.Function to get movies that a user rated and the ratings of those movies 

In [10]:
def get_items_rated_by_user(rate_matrix,user_id):
    y_user=rate_matrix[:,0]
    ids=np.where(y_user==user_id)[0]
    items_id=rate_matrix[ids,1]
    scores=rate_matrix[ids,2]
    return (items_id,scores)

In [11]:
print(get_items_rated_by_user(ratings_train,0)) #user 1

(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  33,  34,  35,  36,  37,  38,  39,  40,
        41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
        54,  55,  56,  57,  58,  59,  61,  62,  63,  64,  65,  66,  67,
        68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,
        81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,
        94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106,
       107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120,
       121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133,
       134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146,
       147, 148, 149, 150, 151, 152, 153, 155, 156, 157, 158, 160, 161,
       162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175,
       176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 1

### 2.3.Build Content-Base Filtering System

**Using Ridge Regression to find weights and bias for each user**

Minimize this function:
$$L(\mathbf{w}_u)= \sum_{i \in R_u} (\mathbf{w}_u \mathbf{x}_i+b_u-y_{i,u})^2 + \lambda||\mathbf{w}_u||^2$$

**Using Linear Function to predict** 

Score-predicted $\hat{y}_{i,u}$ for movie $i$ and user $u$ 
$$\hat{y}_{i,u} = \mathbf{x}_i \cdot \mathbf{w}_u + b_u$$

**Meaning of parameters**
* $\mathbf{x}_i$: tfidf feature vector for movie i
* $\mathbf{w}_u$: weight vector for user u
* $b_u$: bias for user u
* $y_{i,u}$: real score of movie i rating by user u
* $\lambda$: parameter to prevent overfitting

### 2.4.Run and test model

**Define and fit model**

In [12]:
n_users=len(np.unique(ratings_train[:,0]))
print(f'Number of users: {n_users}')

Number of users: 943


In [13]:
model=ContentBasedFiltering(tfidf,n_users,_lambda=10)
model.fit(ratings_train)

**Predict a user**

In [14]:
movies_id,real_ratings,predict_ratings=model.predict_one_user(0,ratings_test) #user 1
print(f'Movies that user rated: {movies_id}')
print(f'Real ratings: {real_ratings}')
print(f'Predict ratings: {predict_ratings}')

Movies that user rated: [20, 33, 61, 117, 155, 160, 171, 189, 202, 265]
Real ratings: [4, 4, 4, 3, 2, 4, 5, 3, 5, 4]
Predict ratings: [4.020506636872614, 3.5059975220440682, 4.061421108770311, 3.0486562175954677, 3.577150243978573, 4.061421108770311, 4.070126534478444, 3.6640611321946506, 3.7354090832606026, 3.330873558378929]


**Predict a user with a movie**


In [15]:
real_rating,predict_rating=model.predict_one_user_item(0,32,ratings_test) #user 1, movie 33
print(f'Real rating: {real_rating}')
print(f'Predict rating: {predict_rating}')

Real rating: 4
Predict rating: 3.5059975220440682


### 2.5.Evaluate model

In [16]:
rmse_test=model.RMSE(ratings_test)
rmse_train=model.RMSE(ratings_train)
print(f'RMSE for train: {rmse_train}')
print(f'RMSE for test: {rmse_test}')

RMSE for train: 0.9704311906578462
RMSE for test: 1.0249830514613054


## 3.Recommend

**Recommend function**

In [17]:
def recommend(user_id,n_items,ratings_predict,ratings_test,top=10):
    user_scores=ratings_predict[:,user_id].copy()
    rated_movies,_=get_items_rated_by_user(ratings_test,user_id)
    user_scores[rated_movies]=-1e9
    recommend_indices=np.argsort(user_scores)[-top:][::-1]
    return recommend_indices

**Load mapping movie data**

In [18]:
movie_mapping=pd.read_csv('../data/processed/movie_mapping.csv',header=0)
movie_mapping

,movie id,movie title
0,0,Toy Story (1995)
1,1,GoldenEye (1995)
2,2,Four Rooms (1995)
3,3,Get Shorty (1995)
4,4,Copycat (1995)
...,...,...
1677,1677,Mat' i syn (1997)
1678,1678,B. Monkey (1998)
1679,1679,Sliding Doors (1998)
1680,1680,You So Crazy (1994)


**Recommend movie**

In [19]:
n_items=X_items_profile.shape[0]
print(f'Number of movies: {n_items}')

Number of movies: 1682


In [20]:
recommend_movies_indices = recommend(0, n_items, model.Yhat, ratings_test, 10) # top 10 moviesrecommended for user 1
filter_movies=movie_mapping.loc[recommend_movies_indices,'movie title']
for index,title in filter_movies.items():
    print(f'Movie ID: {index+1}, Movie Title: {title}')

Movie ID: 7, Movie Title: Twelve Monkeys (1995)
Movie ID: 429, Movie Title: Day the Earth Stood Still, The (1951)
Movie ID: 258, Movie Title: Contact (1997)
Movie ID: 1006, Movie Title: Until the End of the World (Bis ans Ende der Welt) (1991)
Movie ID: 270, Movie Title: Gattaca (1997)
Movie ID: 552, Movie Title: Species (1995)
Movie ID: 434, Movie Title: Forbidden Planet (1956)
Movie ID: 179, Movie Title: Clockwork Orange, A (1971)
Movie ID: 1154, Movie Title: Alphaville (1965)
Movie ID: 175, Movie Title: Brazil (1985)


**Recommend movies based on content of a movie**

In [21]:
recommend_movies_indices=model.recommend_similar_items(0)
filter_movies=movie_mapping.loc[recommend_movies_indices,'movie title']
for index,title in filter_movies.items():
    print(f'Movie ID: {index+1}, Movie Title: {title}')

Movie ID: 422, Movie Title: Aladdin and the King of Thieves (1996)
Movie ID: 1066, Movie Title: Balto (1995)
Movie ID: 1409, Movie Title: Swan Princess, The (1994)
Movie ID: 946, Movie Title: Fox and the Hound, The (1981)
Movie ID: 625, Movie Title: Sword in the Stone, The (1963)
Movie ID: 102, Movie Title: Aristocats, The (1970)
Movie ID: 969, Movie Title: Winnie the Pooh and the Blustery Day (1968)
Movie ID: 1412, Movie Title: Land Before Time III: The Time of the Great Giving (1995) (V)
Movie ID: 1078, Movie Title: Oliver & Company (1988)
Movie ID: 1470, Movie Title: Gumby: The Movie (1995)


## Save model

In [22]:
joblib.dump(model, '../api/models/cb_model.pkl')

['../api/models/cb_model.pkl']